# E3 on Colab: Qwen 7B as the main agent

Step 6 of the pipeline as planned: an LLM (**Qwen2.5-7B-Instruct**, untuned, zero-shot) combines the
three GNN agents' findings into the verdict. It runs on the same 336 test buildings as the rule
baselines (paths 0.893, route 0.877 building macro F1).

**Before you start**
1. *Runtime > Change runtime type*: **L4** or **A100** (the model needs ~15 GB in 16-bit).
2. The Colab secret **`GH_TOKEN`** from the student notebook (notebook access on).
3. Upload **`e3_upload.zip`** (made on the PC: teacher records and IFC files, the split, the three
   trained agents) to **`MyDrive/ada-llm-gnn/`**. The student's records and IFC files are already
   on Drive from the student run.

Results: `MyDrive/ada-llm-gnn/experiments/<time>_eval_gnn_local_Qwen2.5-7B-Instruct_paths/`.

*Status: written and checked offline only; not run on a GPU yet.*

In [ ]:
import subprocess
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.get_device_properties(0).total_memory / 2**30 >= 22, "pick an L4 or an A100 GPU" 

In [ ]:
import os
import subprocess

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ada-llm-gnn"
REPO = "/content/ada-llm-gnn"
if not os.path.exists(REPO):
    url = f"https://{userdata.get('GH_TOKEN')}@github.com/Byomer1021/ada-llm-gnn.git"
    r = subprocess.run(["git", "clone", "-q", url, REPO], capture_output=True, text=True)
    if r.returncode:  # never print the URL: it contains the token
        raise RuntimeError("git clone failed: check the GH_TOKEN secret")
os.chdir(REPO)
subprocess.run(["git", "pull", "-q"], check=True)
for d in ("data", "models", "experiments"):  # the repo's output folders -> Drive
    os.makedirs(f"{DRIVE}/{d}", exist_ok=True)
    if not os.path.islink(d):
        subprocess.run(["rm", "-rf", d], check=True)
        os.symlink(f"{DRIVE}/{d}", d)
zip_path = f"{DRIVE}/e3_upload.zip"
if os.path.exists(zip_path):  # teacher data, split, agents -> into the Drive folders
    subprocess.run(["unzip", "-q", "-o", zip_path, "-d", DRIVE], check=True)
need = ["data/datasets/split.json", "data/llm/teacher_verified.jsonl", "data/llm/student_verified.jsonl",
        "models/door.pt", "models/room.pt", "models/route.pt"]
missing = [p for p in need if not os.path.exists(p)]
assert not missing, f"missing on Drive: {missing}"
print("teacher IFC:", len(os.listdir("data/synthetic/teacher")), "| student IFC:",
      len(os.listdir("data/synthetic/student")), "|",
      subprocess.run(["git", "log", "--format=%h %s", "-1"], capture_output=True, text=True).stdout)

## 1. Install and serve Qwen 7B with vLLM

vLLM first (it brings its own torch), then the pipeline's packages. The first start downloads ~15 GB.

In [ ]:
!pip install -q vllm
!pip install -q ifcopenshell torch_geometric openai pyyaml scikit-learn networkx

In [ ]:
import json
import time
import urllib.request

log = open("/content/vllm.log", "w")
server = subprocess.Popen(["vllm", "serve", "Qwen/Qwen2.5-7B-Instruct", "--max-model-len", "8192",
                           "--gpu-memory-utilization", "0.85", "--port", "8000"],
                          stdout=log, stderr=subprocess.STDOUT)
for _ in range(240):
    try:
        models = json.load(urllib.request.urlopen("http://localhost:8000/v1/models", timeout=2))
        print("ready:", [m["id"] for m in models["data"]])
        break
    except Exception:
        if server.poll() is not None:
            raise RuntimeError("vLLM stopped:\n" + open("/content/vllm.log").read()[-3000:])
        time.sleep(5)
else:
    raise TimeoutError("vLLM not ready after 20 minutes, see /content/vllm.log")

## 2. Pilot: 10 buildings

Checks that the answers parse (look for `invalid` and `<-- wrong`).

In [ ]:
!python scripts/evaluate.py --provider local --llm-url http://localhost:8000/v1 --llm-model Qwen/Qwen2.5-7B-Instruct --workers 8 --limit 10

## 3. E3: all 336 test buildings

Same agents and test split as the rule baselines on the PC. Paste the printed metrics back.

In [ ]:
!python scripts/evaluate.py --provider local --llm-url http://localhost:8000/v1 --llm-model Qwen/Qwen2.5-7B-Instruct --workers 16 | grep -E 'findings for|"(accuracy|macro_f1|invalid_answers|precision|recall|f1|exact_match|accessible|not_accessible)"|saved'

In [ ]:
server.terminate()  # then: Runtime > Disconnect and delete runtime